# BASE MODEL

In [ ]:
# # ===== Install all required dependencies =====

In [ ]:
# !pip install -U pandas numpy scikit-learn
# !pip install -U transformers accelerate
# !pip install -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# !pip install -U tqdm regex

In [3]:
# Toxic Comment Classification Model (Baseline: TF-IDF + Logistic Regression)
# Dataset: Jigsaw Toxic Comment Classification Challenge

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import roc_auc_score, classification_report

In [4]:
# 1. Load data
train_df = pd.read_csv('datasets/train.csv')

LABELS = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

X = train_df['comment_text'].fillna('')
y = train_df[LABELS]

In [5]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    return text

X = X.apply(clean_text)

In [6]:
# 2. Train-validation split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [7]:
# 3. TF-IDF Vectorization
tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1,2),
    stop_words='english'
)
X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)

In [8]:
# 4. Model (One-vs-Rest Logistic Regression)
model = OneVsRestClassifier(
    LogisticRegression(max_iter=1000, solver='liblinear')
)
model.fit(X_train_tfidf, y_train)

,"estimator estimator: estimator objectA regressor or a classifier that implements :term:`fit`.When a classifier is passed, :term:`decision_function` will be usedin priority and it will fallback to :term:`predict_proba` if it is notavailable.When a regressor is passed, :term:`predict` is used.",LogisticRegre...r='liblinear')
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation: the `n_classes`one-vs-rest problems are computed in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: 0.20 `n_jobs` default changed from 1 to None",None
,"verbose verbose: int, default=0The verbosity level, if non zero, progress messages are printed.Below 50, the output is sent to stderr. Otherwise, the output is sentto stdout. The frequency of the messages increases with the verbositylevel, reporting all iterations at 10. See :class:`joblib.Parallel` formore details... versionadded:: 1.1",0
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=

In [9]:
# 5. Evaluation
y_val_pred = model.predict_proba(X_val_tfidf)

roc_auc = roc_auc_score(y_val, y_val_pred, average='macro')
print(f"Macro ROC-AUC: {roc_auc:.4f}")

for i, label in enumerate(LABELS):
    print(f"\nLabel: {label}")
    print(classification_report(y_val[label], (y_val_pred[:, i] > 0.5).astype(int)))

Macro ROC-AUC: 0.9777

Label: toxic
              precision    recall  f1-score   support

           0       0.96      0.99      0.98     28859
           1       0.92      0.59      0.71      3056

    accuracy                           0.96     31915
   macro avg       0.94      0.79      0.85     31915
weighted avg       0.95      0.96      0.95     31915


Label: severe_toxic
              precision    recall  f1-score   support

           0       0.99      1.00      1.00     31594
           1       0.54      0.21      0.30       321

    accuracy                           0.99     31915
   macro avg       0.77      0.61      0.65     31915
weighted avg       0.99      0.99      0.99     31915


Label: obscene
              precision    recall  f1-score   support

           0       0.98      1.00      0.99     30200
           1       0.93      0.60      0.73      1715

    accuracy                           0.98     31915
   macro avg       0.95      0.80      0.86     31915
w

In [10]:
# 6. Train on full data and predict test set

test_df = pd.read_csv('datasets/test.csv')
X_test = test_df['comment_text'].fillna('')
X_test_tfidf = tfidf.transform(X_test)

test_pred = model.predict_proba(X_test_tfidf)

submission = pd.DataFrame(test_pred, columns=LABELS)
submission.insert(0, 'id', test_df['id'])
submission.to_csv('submission.csv', index=False)

print('Submission file saved as submission.csv')

Submission file saved as submission.csv


In [36]:
# 7. Simple CLI Menu for Demonstration
print("=== TF-IDF Toxic Comment Classifier Demo ===")
print("Type a comment and see toxicity predictions.")
print("Type 'exit' to quit.\n")

while True:
    user_input = input("Enter a comment: ")
    if user_input.lower() == 'exit':
        print("Exiting demo.")
        break

    # Echo user input
    print("\nYou entered:")
    print(f"\"{user_input}\"")

    # Clean text (same as training)
    cleaned = user_input.lower()
    cleaned = ''.join([c if c.isalpha() or c.isspace() else ' ' for c in cleaned])

    # Vectorize & predict
    vec = tfidf.transform([cleaned])
    probs = model.predict_proba(vec)[0]

    print("\nPrediction results:")
    for label, prob in zip(LABELS, probs):
        flag = "!!" if prob >= 0.5 else ""
        print(f"{label:15s}: {prob:.3f} {flag}")

    print("\n" + "-" * 40 + "\n")

=== TF-IDF Toxic Comment Classifier Demo ===
Type a comment and see toxicity predictions.
Type 'exit' to quit.


You entered:
"i wanna kill myself"

Prediction results:
toxic          : 0.824 !!
severe_toxic   : 0.043 
obscene        : 0.219 
threat         : 0.644 !!
insult         : 0.118 
identity_hate  : 0.032 

----------------------------------------


You entered:
"i love jason"

Prediction results:
toxic          : 0.070 
severe_toxic   : 0.004 
obscene        : 0.024 
threat         : 0.002 
insult         : 0.012 
identity_hate  : 0.006 

----------------------------------------


You entered:
"these bugs are killing me"

Prediction results:
toxic          : 0.102 
severe_toxic   : 0.006 
obscene        : 0.031 
threat         : 0.003 
insult         : 0.024 
identity_hate  : 0.006 

----------------------------------------


You entered:
""

Prediction results:
toxic          : 0.069 
severe_toxic   : 0.006 
obscene        : 0.025 
threat         : 0.002 
insult         : 0.

# TRANSFORMER-BASED MODEL (DistilBERT)


In [12]:
# This section fine-tunes DistilBERT to address the limitations
# of TF-IDF + Logistic Regression using contextual embeddings.

In [13]:
import torch
from torch.utils.data import Dataset
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import roc_auc_score, f1_score

c:\Users\mikhael\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
# 1. Tokenizer (handles tokenization + normalization)
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')


class ToxicDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding='max_length',
            max_length=max_len
    )
        self.labels = labels.values


    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item


    def __len__(self):
        return len(self.labels)

In [15]:
# 2. Create datasets
train_dataset = ToxicDataset(X_train.tolist(), y_train, tokenizer)
val_dataset = ToxicDataset(X_val.tolist(), y_val, tokenizer)

In [16]:
# 3. Model (multi-label classification)
bert_model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=len(LABELS),
    problem_type='multi_label_classification'
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


The warning relates to an optional optimized download backend and does not affect model training or performance.

In [17]:
# 4. Metrics for evaluation
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.sigmoid(torch.tensor(logits)).numpy()
    roc_auc = roc_auc_score(labels, probs, average='macro')
    preds = (probs > 0.5).astype(int)
    f1 = f1_score(labels, preds, average='macro')
    return {
    'roc_auc': roc_auc,
    'f1': f1
}

In [18]:
import transformers
print(transformers.__version__)
print(transformers.__file__)

4.57.3
c:\Users\mikhael\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\__init__.py


In [19]:
from transformers import TrainingArguments
import inspect

print(inspect.signature(TrainingArguments))

(output_dir: Optional[str] = None, overwrite_output_dir: bool = False, do_train: bool = False, do_eval: bool = False, do_predict: bool = False, eval_strategy: Union[transformers.trainer_utils.IntervalStrategy, str] = 'no', prediction_loss_only: bool = False, per_device_train_batch_size: int = 8, per_device_eval_batch_size: int = 8, per_gpu_train_batch_size: Optional[int] = None, per_gpu_eval_batch_size: Optional[int] = None, gradient_accumulation_steps: int = 1, eval_accumulation_steps: Optional[int] = None, eval_delay: float = 0, torch_empty_cache_steps: Optional[int] = None, learning_rate: float = 5e-05, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, max_grad_norm: float = 1.0, num_train_epochs: float = 3.0, max_steps: int = -1, lr_scheduler_type: Union[transformers.trainer_utils.SchedulerType, str] = 'linear', lr_scheduler_kwargs: Union[dict[str, Any], str] = <factory>, warmup_ratio: float = 0.0, warmup_steps: int = 0, log

In [20]:
pip install transformers[torch]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
pip install --upgrade accelerate

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
import accelerate
print(accelerate.__version__)

1.12.0


In [23]:
# 5. Training configuration
training_args = TrainingArguments(
    output_dir='./bert_results',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=True,
    logging_dir='./bert_logs',
    load_best_model_at_end=True,
    metric_for_best_model='roc_auc'
)

In [24]:
# 6. Trainer
trainer = Trainer(
model=bert_model,
args=training_args,
train_dataset=train_dataset,
eval_dataset=val_dataset,
compute_metrics=compute_metrics
)

In [26]:
import torch
print(torch.cuda.is_available())

True


In [27]:
next(trainer.model.parameters()).device

device(type='cuda', index=0)

In [29]:
# 7. Fine-tuning
trainer.train()

Epoch,Training Loss,Validation Loss,Roc Auc,F1
1,0.038300,0.036443,0.991277,0.605053
2,0.026500,0.038884,0.991121,0.673600
3,0.019700,0.045457,0.989949,0.668481


TrainOutput(global_step=23937, training_loss=0.028342228921457664, metrics={'train_runtime': 6070.771, 'train_samples_per_second': 63.084, 'train_steps_per_second': 3.943, 'total_flos': 2.5587638262226944e+16, 'train_loss': 0.028342228921457664, 'epoch': 3.0})

In [30]:
# 8. Evaluation
bert_eval = trainer.evaluate()
print("DistilBERT Evaluation:", bert_eval)

DistilBERT Evaluation: {'eval_loss': 0.03644338250160217, 'eval_roc_auc': 0.9912774476827474, 'eval_f1': 0.6050531630808625, 'eval_runtime': 98.4481, 'eval_samples_per_second': 324.181, 'eval_steps_per_second': 20.264, 'epoch': 3.0}


In [37]:
# ===== DistilBERT CLI Demo (Echo Input) =====
import torch
import numpy as np

LABELS = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
trainer.model.to(device)
trainer.model.eval()

print("=== DistilBERT Toxic Comment Classifier Demo ===")
print("Type a comment and see toxicity predictions.")
print("Type 'exit' to quit.\n")

while True:
    text = input("Enter a comment: ")
    if text.lower() == "exit":
        print("Exiting demo.")
        break

    # Echo user input
    print("\nYou entered:")
    print(f"\"{text}\"")

    # Tokenize
    inputs = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=256,
        return_tensors="pt"
    ).to(device)

    # Inference
    with torch.no_grad():
        outputs = trainer.model(**inputs)
        logits = outputs.logits
        probs = torch.sigmoid(logits).cpu().numpy()[0]

    print("\nPrediction results:")
    for label, prob in zip(LABELS, probs):
        flag = "!!" if prob >= 0.5 else ""
        print(f"{label:15s}: {prob:.3f} {flag}")

    print("\n" + "-" * 40 + "\n")


=== DistilBERT Toxic Comment Classifier Demo ===
Type a comment and see toxicity predictions.
Type 'exit' to quit.


You entered:
"i wanna kill myself"

Prediction results:
toxic          : 0.758 !!
severe_toxic   : 0.005 
obscene        : 0.024 
threat         : 0.178 
insult         : 0.020 
identity_hate  : 0.002 

----------------------------------------


You entered:
"i love jason"

Prediction results:
toxic          : 0.000 
severe_toxic   : 0.000 
obscene        : 0.000 
threat         : 0.000 
insult         : 0.000 
identity_hate  : 0.000 

----------------------------------------


You entered:
"these bugs are killing me"

Prediction results:
toxic          : 0.369 
severe_toxic   : 0.000 
obscene        : 0.004 
threat         : 0.006 
insult         : 0.012 
identity_hate  : 0.001 

----------------------------------------

Exiting demo.
